In [17]:
import numpy as np

def pretty_matrix(M: np.ndarray) -> str:
    return ",\n".join(", ".join(f"{v:.8f}f" for v in row) for row in M)

# xy → XYZ (Y=1)
def xy_to_XYZ(x: float, y: float) -> np.ndarray:
    X = x / y
    Y = 1.0
    Z = (1.0 - x - y) / y
    return np.array([X, Y, Z], dtype=np.float64)

# RGB(lin) → XYZ
def rgb_to_xyz_matrix(primaries: dict, white_xy: tuple) -> np.ndarray:
    Xr, Yr, Zr = xy_to_XYZ(*primaries['r'])
    Xg, Yg, Zg = xy_to_XYZ(*primaries['g'])
    Xb, Yb, Zb = xy_to_XYZ(*primaries['b'])
    Xw, Yw, Zw = xy_to_XYZ(*white_xy)

    N = np.array([[Xr, Xg, Xb],
                  [Yr, Yg, Yb],
                  [Zr, Zg, Zb]], dtype=np.float64)

    S = np.linalg.solve(N, np.array([Xw, Yw, Zw], dtype=np.float64))
    M = N * S
    return M.astype(np.float32)

# XYZ → RGB(lin)
def xyz_to_rgb_matrix(primaries: dict, white_xy: tuple) -> np.ndarray:
    M = rgb_to_xyz_matrix(primaries, white_xy).astype(np.float64)
    return np.linalg.inv(M).astype(np.float32)

# Primaries (all D65)
D65 = (0.3127, 0.3290)
PRIM_601_625 = { 'r': (0.640, 0.330), 'g': (0.290, 0.600), 'b': (0.150, 0.060) }   # EBU/625
PRIM_601_525 = { 'r': (0.630, 0.340), 'g': (0.310, 0.595), 'b': (0.155, 0.070) }   # SMPTE-C/525
PRIM_709     = { 'r': (0.640, 0.330), 'g': (0.300, 0.600), 'b': (0.150, 0.060) }   # Rec.709 / sRGB
PRIM_2020    = { 'r': (0.708, 0.292), 'g': (0.170, 0.797), 'b': (0.131, 0.046) }   # Rec.2020


In [18]:
def oetf_bt709_encode(v_lin: np.ndarray) -> np.ndarray:
    """Linear -> gamma-encoded (per-channel), piecewise BT.709 OETF."""
    v = np.asarray(v_lin, dtype=np.float64)
    return np.where(v < 0.018, 4.5*v, 1.099*np.power(v, 0.45) - 0.099).astype(np.float64)

def eotf_bt709_decode(v_enc: np.ndarray) -> np.ndarray:
    """Gamma-encoded -> linear (per-channel), inverse BT.709 OETF."""
    v = np.asarray(v_enc, dtype=np.float64)
    return np.where(v < 0.08145, v/4.5, np.power((v + 0.099)/1.099, 1/0.45)).astype(np.float64)


In [19]:
# Build linear RGB_src -> RGB709 matrix (shared D65 white)
M_XYZ_to_709 = xyz_to_rgb_matrix(PRIM_709, D65)

def M_src_to_709_linear(prim_src: dict) -> np.ndarray:
    M_src_to_XYZ = rgb_to_xyz_matrix(prim_src, D65)
    return (M_XYZ_to_709 @ M_src_to_XYZ).astype(np.float32)

# Non-linear helper: R'G'B' (src) -> R'G'B' (709)
def rgbp_src_to_rgbp709(rgbp: np.ndarray, prim_src: dict) -> np.ndarray:
    lin = eotf_bt709_decode(rgbp)                 # SDR assumption for 601/709/2020
    M   = M_src_to_709_linear(prim_src)
    lin709 = (lin @ M.T).clip(0, None)
    return oetf_bt709_encode(lin709).clip(0, 1)

# Show the inside linear matrices you’ll be using:
M_601_625_to_709 = M_src_to_709_linear(PRIM_601_625)
M_601_525_to_709 = M_src_to_709_linear(PRIM_601_525)
M_2020_to_709    = M_src_to_709_linear(PRIM_2020)

print("RGB(lin) 601(625) -> 709 matrix:\n", pretty_matrix(M_601_625_to_709))
print("\nRGB(lin) 601(525) -> 709 matrix:\n", pretty_matrix(M_601_525_to_709))
print("\nRGB(lin) 2020 -> 709 matrix:\n", pretty_matrix(M_2020_to_709))


RGB(lin) 601(625) -> 709 matrix:
 1.04404318f, -0.04404324f, 0.00000000f,
0.00000001f, 1.00000000f, -0.00000001f,
-0.00000000f, 0.01179338f, 0.98820668f

RGB(lin) 601(525) -> 709 matrix:
 0.93954194f, 0.05018133f, 0.01027656f,
0.01777223f, 0.96579289f, 0.01643492f,
-0.00162160f, -0.00436975f, 1.00599146f

RGB(lin) 2020 -> 709 matrix:
 1.66049099f, -0.58764112f, -0.07284993f,
-0.12455052f, 1.13289988f, -0.00834943f,
-0.01815076f, -0.10057890f, 1.11872983f


In [20]:
M_709_to_XYZ = rgb_to_xyz_matrix(PRIM_709, D65)
M_XYZ_to_709 = xyz_to_rgb_matrix(PRIM_709, D65)

print("RGB709(lin) -> XYZ:\n", pretty_matrix(M_709_to_XYZ))
print("\nXYZ -> RGB709(lin):\n", pretty_matrix(M_XYZ_to_709))


RGB709(lin) -> XYZ:
 0.41239080f, 0.35758433f, 0.18048079f,
0.21263900f, 0.71516865f, 0.07219232f,
0.01933082f, 0.11919478f, 0.95053214f

XYZ -> RGB709(lin):
 3.24096990f, -1.53738320f, -0.49861076f,
-0.96924365f, 1.87596750f, 0.04155505f,
0.05563009f, -0.20397697f, 1.05697155f


In [21]:
# Luma weights for 709 (Kr, Kg, Kb)
Kr, Kb = 0.2126, 0.0722
Kg = 1.0 - Kr - Kb

def yuv709_matrices_full():
    """Matrices for R'G'B'709 <-> Y'CbCr709 in FULL range (no offsets)."""
    denom_cb = 2.0 * (1.0 - Kb)
    denom_cr = 2.0 * (1.0 - Kr)

    # RGB' -> YUV' (full)
    M_rgb2yuv = np.array([
        [ Kr,        Kg,        Kb      ],
        [-Kr/denom_cb, -Kg/denom_cb,  0.5     ],
        [ 0.5,      -Kg/denom_cr, -Kb/denom_cr]
    ], dtype=np.float32)

    # YUV' -> RGB' (full) (standard closed-form)
    M_yuv2rgb = np.array([
        [1.0,           0.0,                 2.0*(1.0-Kr)           ],
        [1.0,  -2.0*Kb*(1.0-Kb)/Kg,  -2.0*Kr*(1.0-Kr)/Kg            ],
        [1.0,   2.0*(1.0-Kb),         0.0                           ]
    ], dtype=np.float32)
    return M_rgb2yuv, M_yuv2rgb

def yuv709_matrices_video():
    """
    Studio-range (video) coding:
      Y_code  = 16/255 + (219/255) * Y'
      C*_code = 128/255 + (224/255) * C*'
    Returns (M_rgb2yuv_code, offset_rgb2yuv_code, M_yuvcode2rgb, offset_yuvcode2rgb)
    where M_yuvcode2rgb applies AFTER de-offset/scale.
    """
    alpha = 219.0/255.0   # luma scale
    beta  = 224.0/255.0   # chroma scale
    offY, offC = 16.0/255.0, 128.0/255.0

    M_full_rgb2yuv, M_full_yuv2rgb = yuv709_matrices_full()

    # Code-space matrix is just scaled rows; add offsets separately
    M_rgb2yuv_code = np.diag([alpha, beta, beta]) @ M_full_rgb2yuv
    off_rgb2yuv    = np.array([offY, offC, offC], dtype=np.float32)

    # For inverse, you'll first undo offsets/scales, then apply M_full_yuv2rgb.
    # We return the full matrices plus the needed scales/offsets so usage is explicit.
    return M_rgb2yuv_code.astype(np.float32), off_rgb2yuv, M_full_yuv2rgb.astype(np.float32), np.array([offY, offC, offC], dtype=np.float32)

# Pretty-print the FULL-range matrices:
M_r2y_full, M_y2r_full = yuv709_matrices_full()
print("RGB'709 -> YUV'709 (FULL range):\n", pretty_matrix(M_r2y_full))
print("\nYUV'709 -> RGB'709 (FULL range):\n", pretty_matrix(M_y2r_full))

# And the studio/video (code) matrices:
M_r2y_code, off_r2y, M_ycode2r_full, off_ycode = yuv709_matrices_video()
print("\nRGB'709 -> YUV709 (studio range) MATRIX:\n", pretty_matrix(M_r2y_code))
print("\nRGB'709 -> YUV709 (studio range) OFFSET (Y, Cb, Cr):\n", off_r2y)

# Example usage (studio range):
#  yuv_code = M_r2y_code @ rgbp + off_r2y
#  rgbp     = M_ycode2r_full @ ((yuv_code - off_ycode) / [alpha, beta, beta])


RGB'709 -> YUV'709 (FULL range):
 0.21259999f, 0.71520001f, 0.07220000f,
-0.11457211f, -0.38542789f, 0.50000000f,
0.50000000f, -0.45415291f, -0.04584709f

YUV'709 -> RGB'709 (FULL range):
 1.00000000f, 0.00000000f, 1.57480001f,
1.00000000f, -0.18732427f, -0.46812427f,
1.00000000f, 1.85560000f, 0.00000000f

RGB'709 -> YUV709 (studio range) MATRIX:
 0.18258588f, 0.61423057f, 0.06200706f,
-0.10064373f, -0.33857197f, 0.43921569f,
0.43921569f, -0.39894217f, -0.04027352f

RGB'709 -> YUV709 (studio range) OFFSET (Y, Cb, Cr):
 [0.0627451 0.5019608 0.5019608]


In [33]:
OFF_YUV_CODE = np.array([16/255, 128/255, 128/255], dtype=np.float32)
S_CODE_TO_FULL = np.array([255/219, 255/224, 255/224], dtype=np.float32)  # element-wise
S_FULL_TO_CODE = np.array([219/255, 224/255, 224/255], dtype=np.float32)

def yuv_code_to_full(yuv_code: np.ndarray) -> np.ndarray:
    return (np.asarray(yuv_code, float) - OFF_YUV_CODE) * S_CODE_TO_FULL

def yuv_full_to_code(yuv_full: np.ndarray) -> np.ndarray:
    return np.asarray(yuv_full, float) * S_FULL_TO_CODE + OFF_YUV_CODE

In [34]:
def rgbp_to_yuvp_full_matrix_from_kr_kb(Kr: float, Kb: float) -> np.ndarray:
    Kg = 1.0 - Kr - Kb
    denom_cb = 2.0 * (1.0 - Kb)
    denom_cr = 2.0 * (1.0 - Kr)
    return np.array([
        [ Kr,             Kg,              Kb          ],
        [ -Kr/denom_cb,  -Kg/denom_cb,     0.5         ],
        [  0.5,          -Kg/denom_cr,    -Kb/denom_cr ]
    ], dtype=np.float32)

def yuvp_to_rgbp_full_matrix_from_kr_kb(Kr: float, Kb: float) -> np.ndarray:
    Kg = 1.0 - Kr - Kb
    return np.array([
        [ 1.0,                      0.0,                       2.0*(1.0-Kr)           ],
        [ 1.0,   -2.0*Kb*(1.0-Kb)/Kg,      -2.0*Kr*(1.0-Kr)/Kg                        ],
        [ 1.0,                      2.0*(1.0-Kb),              0.0                     ]
    ], dtype=np.float32)

In [35]:
def yuv_code_to_rgbp_affine_matrix_from_kr_kb(Kr: float, Kb: float) -> np.ndarray:
    """
    Returns A (3x3) such that:
        rgb' = A @ (yuv_code - OFF_YUV_CODE)
    where yuv_code is studio/code in [0,1].
    """
    M_yuv2rgb_full = yuvp_to_rgbp_full_matrix_from_kr_kb(Kr, Kb)
    S_inv = np.diag(S_CODE_TO_FULL.astype(np.float32))  # code -> full scales
    return (M_yuv2rgb_full @ S_inv).astype(np.float32)

In [36]:
def yuv601_525_full_y2r():   return yuvp_to_rgbp_full_matrix_from_kr_kb(0.2990, 0.1140)
def yuv601_625_full_y2r():   return yuvp_to_rgbp_full_matrix_from_kr_kb(0.2990, 0.1140)
def yuv709_full_y2r():       return yuvp_to_rgbp_full_matrix_from_kr_kb(0.2126, 0.0722)
def yuv2020_full_y2r():      return yuvp_to_rgbp_full_matrix_from_kr_kb(0.2627, 0.0593)

def yuv601_525_code_A():     return yuv_code_to_rgbp_affine_matrix_from_kr_kb(0.2990, 0.1140)
def yuv601_625_code_A():     return yuv_code_to_rgbp_affine_matrix_from_kr_kb(0.2990, 0.1140)
def yuv709_code_A():         return yuv_code_to_rgbp_affine_matrix_from_kr_kb(0.2126, 0.0722)
def yuv2020_code_A():        return yuv_code_to_rgbp_affine_matrix_from_kr_kb(0.2627, 0.0593)

In [37]:
print("YUV' 601_525 (FULL) → RGB'  matrix:\n", pretty_matrix(yuv601_525_full_y2r()))
print("\nYUV 601_525 (STUDIO code) → RGB'  A:\n", pretty_matrix(yuv601_525_code_A()))
print("Offset to subtract (Y, Cb, Cr):", OFF_YUV_CODE)

print("\n=========================\n")

print("YUV' 601_625 (FULL) → RGB'  matrix:\n", pretty_matrix(yuv601_625_full_y2r()))
print("\nYUV 601_625 (STUDIO code) → RGB'  A:\n", pretty_matrix(yuv601_625_code_A()))
print("Offset to subtract (Y, Cb, Cr):", OFF_YUV_CODE)

print("\n=========================\n")

print("YUV' 709 (FULL) → RGB'  matrix:\n", pretty_matrix(yuv709_full_y2r()))
print("\nYUV 709 (STUDIO code) → RGB'  A:\n", pretty_matrix(yuv709_code_A()))
print("Offset to subtract (Y, Cb, Cr):", OFF_YUV_CODE)

print("\n=========================\n")

print("YUV' 2020 (FULL) → RGB'  matrix:\n", pretty_matrix(yuv2020_full_y2r()))
print("\nYUV 2020 (STUDIO code) → RGB'  A:\n", pretty_matrix(yuv2020_code_A()))
print("Offset to subtract (Y, Cb, Cr):", OFF_YUV_CODE)

YUV' 601_525 (FULL) → RGB'  matrix:
 1.00000000f, 0.00000000f, 1.40199995f,
1.00000000f, -0.34413630f, -0.71413630f,
1.00000000f, 1.77199996f, 0.00000000f

YUV 601_525 (STUDIO code) → RGB'  A:
 1.16438353f, 0.00000000f, 1.59602666f,
1.16438353f, -0.39176229f, -0.81296766f,
1.16438353f, 2.01723194f, 0.00000000f
Offset to subtract (Y, Cb, Cr): [0.0627451 0.5019608 0.5019608]


YUV' 601_625 (FULL) → RGB'  matrix:
 1.00000000f, 0.00000000f, 1.40199995f,
1.00000000f, -0.34413630f, -0.71413630f,
1.00000000f, 1.77199996f, 0.00000000f

YUV 601_625 (STUDIO code) → RGB'  A:
 1.16438353f, 0.00000000f, 1.59602666f,
1.16438353f, -0.39176229f, -0.81296766f,
1.16438353f, 2.01723194f, 0.00000000f
Offset to subtract (Y, Cb, Cr): [0.0627451 0.5019608 0.5019608]


YUV' 709 (FULL) → RGB'  matrix:
 1.00000000f, 0.00000000f, 1.57480001f,
1.00000000f, -0.18732427f, -0.46812427f,
1.00000000f, 1.85560000f, 0.00000000f

YUV 709 (STUDIO code) → RGB'  A:
 1.16438353f, 0.00000000f, 1.79274106f,
1.16438353f, -0.213